In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import INTERIM_DATA, CLEAN_DATA

In [2]:
df = pd.read_parquet(INTERIM_DATA / '03-data-merge-feature.parquet')

In [3]:
print(df.columns.to_list(), f'\n{df['date']}')

['date', 'expiration', 'T', 'k', 'iv', 'spy_ret', 'delta', 'gamma', 'theta', 'vega', 'rho', 'vix', 'vix_lag', 'vix_mom', 'vix_mom_lag', 'iv_lag', 'log_moneyness', 'd_iv_lag', 'log_oi', 'log_volume', 'abs_spy_ret', 'ret_over_sqrtT', 'd_iv'] 
0         2013-01-03
1         2013-01-03
2         2013-01-03
3         2013-01-03
4         2013-01-03
             ...    
3788636   2026-01-30
3788637   2026-01-30
3788638   2026-01-30
3788639   2026-01-30
3788640   2026-01-30
Name: date, Length: 3788641, dtype: datetime64[ns]


## 1. Data Range

In [4]:
ranges = {
    'A': ('2013-01-03', '2026-01-30'),  # All data
    'B': ('2013-01-03', '2020-02-19'),  # Pre-COVID
    'C': ('2020-03-23', '2026-01-30'),  # Post-COVID
    'D': ('2023-01-01', '2026-01-30'),  # Most recent
}

## 2. Random Split

In [5]:
from sklearn.model_selection import train_test_split

for tag, (start, end) in ranges.items():
    sub = df[(df['date'] >= start) & (df['date'] <= end)].copy()
    trn_val, tst = train_test_split(sub, test_size=0.10, random_state=42)
    trn, val = train_test_split(trn_val, test_size=2/9, random_state=42)
    globals()[f'df_rand_{tag}_train'] = trn
    globals()[f'df_rand_{tag}_val']   = val
    globals()[f'df_rand_{tag}_test']  = tst

## 3. Chronological Split

In [6]:
for tag, (start, end) in ranges.items():
    sub = df[(df['date'] >= start) & (df['date'] <= end)].sort_values('date').reset_index(drop=True)
    n = len(sub)
    t1, t2 = int(n * 0.70), int(n * 0.90)
    globals()[f'df_chro_{tag}_train'] = sub.iloc[:t1]
    globals()[f'df_chro_{tag}_val']   = sub.iloc[t1:t2]
    globals()[f'df_chro_{tag}_test']  = sub.iloc[t2:]

## 4. Save data

In [7]:
for tag in ranges:
    for method in ['rand', 'chro']:
        for split in ['train', 'val', 'test']:
            name = f'df_{method}_{tag}_{split}'
            filename = f'{method}_{tag}_{split}.parquet'
            globals()[name].to_parquet(CLEAN_DATA / filename)
            print(f'Saved {filename}  ({len(globals()[name]):,} rows)')

Saved rand_A_train.parquet  (2,652,048 rows)
Saved rand_A_val.parquet  (757,728 rows)
Saved rand_A_test.parquet  (378,865 rows)
Saved chro_A_train.parquet  (2,652,048 rows)
Saved chro_A_val.parquet  (757,728 rows)
Saved chro_A_test.parquet  (378,865 rows)
Saved rand_B_train.parquet  (911,172 rows)
Saved rand_B_val.parquet  (260,336 rows)
Saved rand_B_test.parquet  (130,168 rows)
Saved chro_B_train.parquet  (911,173 rows)
Saved chro_B_val.parquet  (260,335 rows)
Saved chro_B_test.parquet  (130,168 rows)
Saved rand_C_train.parquet  (1,716,781 rows)
Saved rand_C_val.parquet  (490,509 rows)
Saved rand_C_test.parquet  (245,255 rows)
Saved chro_C_train.parquet  (1,716,781 rows)
Saved chro_C_val.parquet  (490,509 rows)
Saved chro_C_test.parquet  (245,255 rows)
Saved rand_D_train.parquet  (843,153 rows)
Saved rand_D_val.parquet  (240,901 rows)
Saved rand_D_test.parquet  (120,451 rows)
Saved chro_D_train.parquet  (843,153 rows)
Saved chro_D_val.parquet  (240,901 rows)
Saved chro_D_test.parquet 